In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import requests


def fetch_live_headlines(topic: str = "financial markets", limit: int = 20) -> pd.DataFrame:
    """Fetches real-time headlines from Google News RSS for any keyword/topic."""
    formatted_topic = topic.replace(" ", "%20")
    rss_url = f"https://news.google.com/rss/search?q={formatted_topic}&hl=en-US&gl=US&ceid=US:en"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    response = requests.get(rss_url, headers=headers, timeout=10)
    if response.status_code != 200:
        raise ConnectionError(f"Failed to fetch RSS feed: Status {response.status_code}")

    root = ET.fromstring(response.content)
    articles = []

    for item in root.findall(".//item")[:limit]:
        title = item.find("title").text if item.find("title") is not None else ""
        pub_date = item.find("pubDate").text if item.find("pubDate") is not None else ""
        link = item.find("link").text if item.find("link") is not None else ""

        articles.append({"headline": title, "published_at": pub_date, "source_url": link})

    df = pd.DataFrame(articles)
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    return df


# Quick sanity check inside notebook
test_df = fetch_live_headlines("Artificial Intelligence", limit=5)
test_df[["headline", "published_at"]]

,headline,published_at
0,Mark Zuckerberg Blasts Centralization of A.I. ...,2026-07-29 00:59:08
1,Trump administration bans new Chinese humanoid...,2026-07-28 19:40:00
2,Texas A&M University joins the Genesis Mission...,2026-07-28 10:04:36
3,OpenAI's rogue models roamed the internet for ...,2026-07-29 00:57:00
4,EXCLUSIVE: OpenAI's rogue agent compromised a ...,2026-07-28 21:27:00


In [ ]:
import torch
from transformers import pipeline


class SentimentPipeline:

    def __init__(self, model_name: str = "cardiffnlp/twitter-roberta-base-sentiment-latest"):
        """Initializes PyTorch Hugging Face pipeline."""
        self.device = 0 if torch.cuda.is_available() else -1
        print(f"Loading transformer model '{model_name}' on device {self.device}...")

        self.classifier = pipeline(
            task="sentiment-analysis",
            model=model_name,
            tokenizer=model_name,
            device=self.device,
        )

    def predict_dataframe(self, df: pd.DataFrame, text_column: str = "headline") -> pd.DataFrame:
        """Processes an entire DataFrame column through the Transformer model."""
        if df.empty or text_column not in df.columns:
            return df

        texts = df[text_column].tolist()
        results = self.classifier(texts)

        # Standardize labels (positive, neutral, negative)
        sentiments = [res["label"].lower() for res in results]
        confidences = [round(res["score"], 4) for res in results]

        df_out = df.copy()
        df_out["sentiment"] = sentiments
        df_out["confidence_score"] = confidences
        return df_out


# Initialize engine
nlp_engine = SentimentPipeline()

Loading transformer model 'cardiffnlp/twitter-roberta-base-sentiment-latest' on device -1...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 1. Fetch live market news
query_topic = "Tech Stocks"
news_df = fetch_live_headlines(topic=query_topic, limit=20)

# 2. Run Transformer inference
analyzed_df = nlp_engine.predict_dataframe(news_df, text_column="headline")

# Display top 10 results directly in the notebook output
analyzed_df[["headline", "sentiment", "confidence_score"]].head(10)

,headline,sentiment,confidence_score
0,Tech Stocks Tumble on Worries Over A.I. Spendi...,neutral,0.7092
1,Investors Have Been Giving Tech Stocks a Hard ...,neutral,0.4944
2,A Tired Tech Trade Looks Set to Sink the Stock...,negative,0.6831
3,South Korean stock market at three-month low a...,neutral,0.5653
4,U.S. and Korean tech stocks are now tightly li...,neutral,0.6949
5,Why are tech stocks tanking? - Yahoo Finance,negative,0.7275
6,Chip stocks tumble as AI sell-off deepens - Fi...,negative,0.6723
7,U.S. Stocks Move | Leading Tech Stocks Extend ...,neutral,0.6092
8,CXMT: China’s chipmakers trigger bloodbath in ...,negative,0.5456
9,AI Greed Turns Into Fear as Tech Stocks Keep S...,negative,0.7372


In [ ]:
import plotly.express as px
import plotly.io as pio

# Switch to notebook_connected renderer
pio.renderers.default = "notebook_connected"

# 1. Prepare data
sentiment_counts = analyzed_df["sentiment"].value_counts().reset_index()
sentiment_counts.columns = ["Sentiment", "Count"]

# 2. Build Donut Chart
fig_pie = px.pie(
    sentiment_counts,
    values="Count",
    names="Sentiment",
    title=f"Market Sentiment Distribution: '{query_topic}'",
    hole=0.4,
    color="Sentiment",
    color_discrete_map={
        "positive": "#00CC96",
        "neutral": "#636EFA",
        "negative": "#EF553B",
    },
)

fig_pie.show()

In [ ]:
# 1. Filter and sort for the Most Positive headlines
most_positive = analyzed_df[analyzed_df["sentiment"] == "positive"]
most_positive = most_positive.sort_values(by="confidence_score", ascending=False).head(3)

# 2. Filter and sort for the Most Negative headlines
most_negative = analyzed_df[analyzed_df["sentiment"] == "negative"]
most_negative = most_negative.sort_values(by="confidence_score", ascending=False).head(3)

# 3. Display the results clearly in the notebook
print("🟢 TOP 3 POSITIVE CATALYSTS:")
display(most_positive[["headline", "confidence_score"]])

print("\n🔴 TOP 3 NEGATIVE CATALYSTS:")
display(most_negative[["headline", "confidence_score"]])

🟢 TOP 3 POSITIVE CATALYSTS:


,headline,confidence_score
13,Tech stocks today: Big Tech earnings this week...,0.9118
19,S&P 500 ends higher as investors await tech ea...,0.5129



🔴 TOP 3 NEGATIVE CATALYSTS:


,headline,confidence_score
9,AI Greed Turns Into Fear as Tech Stocks Keep S...,0.7372
5,Why are tech stocks tanking? - Yahoo Finance,0.7275
2,A Tired Tech Trade Looks Set to Sink the Stock...,0.6831


In [ ]:
# 1. Generate a clean filename based on your topic
filename = f"sentiment_{query_topic.replace(' ', '_').lower()}.csv"

# 2. Export to CSV (index=False prevents Pandas from adding extra row numbers)
analyzed_df.to_csv(filename, index=False)

print(f"✅ Successfully exported {len(analyzed_df)} rows to '{filename}'!")

✅ Successfully exported 20 rows to 'sentiment_tech_stocks.csv'!
